# Problem 2 — Transformer from Scratch

**Configuration:** model width `d=128`; Task A uses 8 attention heads; Task B uses 4 heads and 4 layers; random seeds `[42, 123, 2026]`.

This notebook is designed to run from top to bottom in Google Colab with a CUDA T4 GPU. It includes:
- scaled dot-product attention, unit tests, and a hand-computed three-token example;
- multi-head attention, a pre-LayerNorm encoder block, sinusoidal positional encoding, and parameter-count verification;
- Task A: UCI HAR encoder classification, with no/sinusoidal/learned positional information;
- Task B: Tiny Shakespeare character-level causal language modelling and temperature-controlled generation;
- attention heatmaps, head-count and no-causal-mask ablations, and context-length time/memory measurements.

The notebook downloads public datasets when needed and writes figures/results using relative paths. Runtime depends on Colab GPU availability. Run **Runtime → Restart session and run all** before submission.

## 0. Environment, configuration and reproducibility

In [ ]:
import os, math, time, random, copy, urllib.request, zipfile, shutil, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SEEDS = [42, 123, 2026]
D_MODEL = 128
TASK_A_HEADS = 8
TASK_B_HEADS = 4
TASK_B_LAYERS = 4
D_FF = 4 * D_MODEL
DROPOUT = 0.1
BATCH_SIZE_HAR = 128
TASK_A_EPOCHS = 10
TASK_B_CONTEXT = 128
TASK_B_BATCH_SIZE = 64
TASK_B_STEPS_PER_EPOCH = 200
TASK_B_EPOCHS = 5
LEARNING_RATE = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Path("problem2/figures").mkdir(parents=True, exist_ok=True)
Path("data").mkdir(exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Seeds:", SEEDS)
print("d =", D_MODEL, "| Task A heads =", TASK_A_HEADS,
      "| Task B heads =", TASK_B_HEADS, "| Task B layers =", TASK_B_LAYERS)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Task 1 — Scaled dot-product attention from scratch

For queries `Q`, keys `K`, and values `V`:

\[
\mathrm{Attention}(Q,K,V)=
\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V.
\]

The optional boolean mask uses `True` for allowed positions and `False` for blocked positions. The function supports leading batch/head dimensions.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None, return_weights=True):
    # Q: (..., Lq, dk), K: (..., Lk, dk), V: (..., Lk, dv)
    dk = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(dk)
    if mask is not None:
        mask = mask.to(device=scores.device, dtype=torch.bool)
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
    weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    if return_weights:
        return output, weights
    return output

### 1.1 Unit tests

In [ ]:
set_seed(42)
Q_test = torch.randn(2, 4, 7, 16, device=DEVICE)
K_test = torch.randn(2, 4, 7, 16, device=DEVICE)
V_test = torch.randn(2, 4, 7, 16, device=DEVICE)

out_custom, weights_custom = scaled_dot_product_attention(Q_test, K_test, V_test)
assert torch.allclose(weights_custom.sum(dim=-1),
                      torch.ones_like(weights_custom.sum(dim=-1)), atol=1e-6)

out_reference = F.scaled_dot_product_attention(Q_test, K_test, V_test, dropout_p=0.0)
sdpa_max_diff = (out_custom - out_reference).abs().max().item()
assert sdpa_max_diff < 1e-5, f"SDPA mismatch: {sdpa_max_diff}"

L = 6
causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=DEVICE))
Qc = torch.randn(1, 2, L, 8, device=DEVICE)
Kc = torch.randn(1, 2, L, 8, device=DEVICE)
Vc = torch.randn(1, 2, L, 8, device=DEVICE)
_, causal_weights = scaled_dot_product_attention(Qc, Kc, Vc, mask=causal)
future = torch.triu(causal_weights, diagonal=1)
assert torch.allclose(future, torch.zeros_like(future), atol=1e-7)

print("Row-sum test: PASS")
print("PyTorch SDPA max absolute difference:", sdpa_max_diff)
print("Causal future-weight test: PASS")

### 1.2 Hand-checkable three-token example

We choose three 2-dimensional token vectors and set `Q = K = V`:

\[
Q=K=V=\begin{bmatrix}1&0\\0&1\\1&1\end{bmatrix},\quad d_k=2.
\]

The scaled score matrix is

\[
\frac{QK^T}{\sqrt 2}=
\begin{bmatrix}
0.7071&0&0.7071\\
0&0.7071&0.7071\\
0.7071&0.7071&1.4142
\end{bmatrix}.
\]

Softmax is applied independently to each row. The resulting attention weights and output are printed below by the code; the rounded expected output is checked against the implementation.

In [ ]:
Q_hand = torch.tensor([[[1., 0.], [0., 1.], [1., 1.]]], device=DEVICE)
K_hand = Q_hand.clone()
V_hand = Q_hand.clone()

scores_hand = Q_hand @ K_hand.transpose(-2, -1) / math.sqrt(2)
out_hand, weights_hand = scaled_dot_product_attention(Q_hand, K_hand, V_hand)

manual_rounded = torch.tensor(
    [[0.8022, 0.5989], [0.5989, 0.8022], [0.7518, 0.7518]],
    device=DEVICE
)
hand_diff = (out_hand[0] - manual_rounded).abs().max().item()

print("Q = K = V:\n", Q_hand[0].cpu())
print("\nScaled scores:\n", scores_hand[0].cpu())
print("\nAttention weights:\n", weights_hand[0].cpu())
print("\nAttention output:\n", out_hand[0].cpu())
print("\nMax difference from hand calculation rounded to 4 decimals:", hand_diff)
assert hand_diff < 5e-4
print("Three-token hand example: PASS")

## 2. Task 2 — Multi-head attention, encoder block and positional encoding

Multi-head attention projects the input into queries, keys and values, splits the model dimension across heads, applies scaled dot-product attention independently, concatenates the heads and applies an output projection.

The encoder block uses **pre-LayerNorm**:
1. `x = x + Dropout(MHA(LN(x)))`
2. `x = x + Dropout(FFN(LN(x)))`

The feed-forward network has width `4d`.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1, causal=False):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.causal = causal
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, need_weights=False):
        # x: (B, L, d); projected tensors: (B, H, L, d_head)
        B, L, _ = x.shape
        q = self.W_Q(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_K(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_V(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)

        effective_mask = mask
        if self.causal and effective_mask is None:
            effective_mask = torch.tril(
                torch.ones(L, L, dtype=torch.bool, device=x.device)
            )

        attended, weights = scaled_dot_product_attention(q, k, v, effective_mask)
        attended = self.attn_dropout(attended)
        # Merge heads: (B, H, L, d_head) -> (B, L, d)
        merged = attended.transpose(1, 2).contiguous().view(B, L, self.d_model)
        output = self.W_O(merged)
        if need_weights:
            return output, weights
        return output


class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, need_weights=False, mask=None):
        if need_weights:
            attn_out, weights = self.attn(
                self.ln1(x), mask=mask, need_weights=True
            )
        else:
            attn_out = self.attn(self.ln1(x), mask=mask)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.ffn(self.ln2(x)))
        if need_weights:
            return x, weights
        return x


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

### 2.1 Shape, gradient and parameter-count checks

In [ ]:
set_seed(42)
test_block = TransformerEncoderBlock(D_MODEL, TASK_A_HEADS, D_FF).to(DEVICE)
x_demo = torch.randn(3, 11, D_MODEL, device=DEVICE, requires_grad=True)
y_demo, w_demo = test_block(x_demo, need_weights=True)
assert y_demo.shape == (3, 11, D_MODEL)
assert w_demo.shape == (3, TASK_A_HEADS, 11, 11)
y_demo.mean().backward()
assert x_demo.grad is not None and torch.isfinite(x_demo.grad).all()

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

block_params = count_parameters(test_block)
# Per-block analytical count:
# MHA: 4*d^2 weights + 4*d biases
# FFN: 2*d*d_ff weights + d_ff + d biases
# LayerNorms: 4*d scale/bias parameters
analytical_block_params = (
    4 * D_MODEL**2 + 4 * D_MODEL
    + 2 * D_MODEL * D_FF + D_FF + D_MODEL
    + 4 * D_MODEL
)
print("Block output shape:", tuple(y_demo.shape))
print("Attention weights shape:", tuple(w_demo.shape))
print("Trainable parameters (code):", block_params)
print("Trainable parameters (analytical):", analytical_block_params)
assert block_params == analytical_block_params
print("Encoder block shape, gradient and parameter tests: PASS")

**Parameter derivation.** The weight-only expression is \(4d^2 + 2d d_{ff}\). For `d=128`, `d_ff=512`, this is 196,608 weights per block. The implementation also has projection/FFN biases and two LayerNorm scale/bias pairs; the assertion above reconciles the full trainable count with `sum(p.numel())`.

## 3. Task A — Transformer encoder classifier on UCI HAR

We reuse the UCI HAR dataset. Each 128-step sequence is projected from 9 sensor channels to `d=128`, a learnable CLS token is prepended, and a two-block Transformer encoder produces a six-class prediction from the CLS representation.

We compare three positional-information settings:
- `none`: no positional encoding;
- `sinusoidal`: fixed sinusoidal encoding;
- `learned`: trainable positional embeddings.

The validation split is subject-wise: four of the 21 training subjects are held out using each selected seed. Normalisation statistics are calculated on the training subjects only.

In [ ]:
# Download and extract UCI HAR if it is not already available.
HAR_ROOT = Path("data/har/UCI HAR Dataset")
if not HAR_ROOT.exists():
    HAR_ROOT.parent.mkdir(parents=True, exist_ok=True)
    archive = Path("data/har.zip")
    url = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
    if not archive.exists():
        urllib.request.urlretrieve(url, archive)
    extract_dir = Path("data/har")
    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(extract_dir)
    nested = extract_dir / "UCI HAR Dataset.zip"
    if nested.exists() and not HAR_ROOT.exists():
        with zipfile.ZipFile(nested, "r") as zf:
            zf.extractall(extract_dir)

assert HAR_ROOT.exists(), f"HAR directory not found: {HAR_ROOT}"

CHANNELS = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]

def load_har_part(split):
    base = HAR_ROOT / split / "Inertial Signals"
    arrays = []
    for channel in CHANNELS:
        path = base / f"{channel}_{split}.txt"
        arrays.append(np.loadtxt(path, dtype=np.float32))
    X = np.stack(arrays, axis=-1)
    y = np.loadtxt(HAR_ROOT / split / f"y_{split}.txt", dtype=np.int64)
    subjects = np.loadtxt(HAR_ROOT / split / f"subject_{split}.txt", dtype=np.int64)
    return X, y, subjects

X_har_train, y_har_train, subj_har_train = load_har_part("train")
X_har_test, y_har_test, subj_har_test = load_har_part("test")
assert X_har_train.shape == (7352, 128, 9)
assert X_har_test.shape == (2947, 128, 9)
print("HAR train/test:", X_har_train.shape, X_har_test.shape)

In [ ]:
def prepare_har_for_seed(seed):
    unique_subjects = np.unique(subj_har_train)
    val_subjects = np.random.default_rng(seed).choice(unique_subjects, 4, replace=False)
    train_mask = ~np.isin(subj_har_train, val_subjects)
    val_mask = np.isin(subj_har_train, val_subjects)

    Xtr, ytr = X_har_train[train_mask], y_har_train[train_mask] - 1
    Xva, yva = X_har_train[val_mask], y_har_train[val_mask] - 1
    Xte, yte = X_har_test, y_har_test - 1

    mean = Xtr.mean(axis=(0, 1), keepdims=True)
    std = Xtr.std(axis=(0, 1), keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)

    Xtr = (Xtr - mean) / std
    Xva = (Xva - mean) / std
    Xte = (Xte - mean) / std

    return (Xtr, ytr), (Xva, yva), (Xte, yte)

def har_loader(pair, shuffle=False, seed=0):
    X, y = pair
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.long))
    gen = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=BATCH_SIZE_HAR,
                      shuffle=shuffle, generator=gen)

class HARTransformerClassifier(nn.Module):
    def __init__(self, d_model=128, num_heads=8, num_layers=2,
                 num_classes=6, positional="sinusoidal", dropout=0.1):
        super().__init__()
        self.positional = positional
        self.input_projection = nn.Linear(9, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embedding = (
            nn.Parameter(torch.zeros(1, 129, d_model))
            if positional == "learned" else None
        )
        self.sinusoidal = (
            SinusoidalPositionalEncoding(d_model, max_len=129)
            if positional == "sinusoidal" else None
        )
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, 4*d_model, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)
        nn.init.normal_(self.cls_token, std=0.02)
        if self.pos_embedding is not None:
            nn.init.normal_(self.pos_embedding, std=0.02)

    def forward(self, x):
        # x: (B, 128, 9) -> token embeddings: (B, 128, d)
        z = self.input_projection(x)
        cls = self.cls_token.expand(z.size(0), -1, -1)
        z = torch.cat([cls, z], dim=1)  # (B, 129, d)
        if self.positional == "sinusoidal":
            z = self.sinusoidal(z)
        elif self.positional == "learned":
            z = z + self.pos_embedding[:, :z.size(1)]
        for block in self.blocks:
            z = block(z)
        return self.classifier(self.final_norm(z[:, 0]))

In [ ]:
def evaluate_classifier(model, loader):
    model.eval()
    ys, preds = [], []
    total_loss, total_n = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb, reduction="sum")
            total_loss += loss.item()
            total_n += yb.numel()
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            ys.extend(yb.cpu().numpy())
    return (total_loss / total_n,
            accuracy_score(ys, preds),
            f1_score(ys, preds, average="macro"),
            np.asarray(ys), np.asarray(preds))

def train_har_transformer(positional, seed, epochs=TASK_A_EPOCHS):
    set_seed(seed)
    train_pair, val_pair, test_pair = prepare_har_for_seed(seed)
    tr_loader = har_loader(train_pair, shuffle=True, seed=seed)
    va_loader = har_loader(val_pair)
    te_loader = har_loader(test_pair)

    model = HARTransformerClassifier(
        d_model=D_MODEL, num_heads=TASK_A_HEADS, num_layers=2,
        positional=positional, dropout=DROPOUT
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_f1 = -1
    best_state = None
    epoch_times = []
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        start = time.perf_counter()
        running_loss, count = 0.0, 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * yb.size(0)
            count += yb.size(0)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        epoch_times.append(time.perf_counter() - start)

        train_loss = running_loss / count
        val_loss, val_acc, val_f1, _, _ = evaluate_classifier(model, va_loader)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())

        print(f"{positional} seed={seed} epoch={epoch+1}/{epochs} "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
              f"val_macroF1={val_f1:.4f}")

    model.load_state_dict(best_state)
    test_loss, test_acc, test_f1, y_true, y_pred = evaluate_classifier(model, te_loader)

    result = {
        "problem": "P2", "model": f"Transformer-HAR-{positional}",
        "seed": seed, "seq_len": 128, "test_acc": test_acc,
        "macro_f1": test_f1, "test_mse": np.nan,
        "params": count_parameters(model),
        "sec_per_epoch": float(np.mean(epoch_times)),
        "val_loss_nats": val_losses[-1], "bits_per_char": np.nan,
        "notes": f"Task A positional={positional}; best_val_macro_f1={best_val_f1:.6f}"
    }
    return result, model, (train_losses, val_losses), (y_true, y_pred)

### 3.1 Train the three positional variants with three seeds

This is the main Task A experiment: nine runs. `TASK_A_EPOCHS` is configurable near the top of the notebook; keep the selected value fixed across all variants and seeds.

In [ ]:
taskA_results = []
taskA_curves = {}
taskA_models = {}
taskA_predictions = {}

for positional in ["none", "sinusoidal", "learned"]:
    for seed in SEEDS:
        result, model, curves, predictions = train_har_transformer(positional, seed)
        taskA_results.append(result)
        taskA_curves[(positional, seed)] = curves
        taskA_models[(positional, seed)] = model
        taskA_predictions[(positional, seed)] = predictions

taskA_df = pd.DataFrame(taskA_results)
display(taskA_df[["model", "seed", "test_acc", "macro_f1", "params", "sec_per_epoch"]])

In [ ]:
taskA_summary = (
    taskA_df.groupby("model")
    .agg(
        test_acc_mean=("test_acc", "mean"),
        test_acc_std=("test_acc", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        params=("params", "first"),
        sec_per_epoch=("sec_per_epoch", "mean")
    ).reset_index()
)
display(taskA_summary.round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
for positional in ["none", "sinusoidal", "learned"]:
    curves = [taskA_curves[(positional, s)] for s in SEEDS]
    mean_train = np.mean([c[0] for c in curves], axis=0)
    mean_val = np.mean([c[1] for c in curves], axis=0)
    ax.plot(mean_train, label=f"{positional} train")
    ax.plot(mean_val, linestyle="--", label=f"{positional} validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Problem 2 Task A — HAR Transformer learning curves")
ax.legend(ncol=2, fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("problem2/figures/taskA_learning_curves.png", dpi=200)
plt.show()

In [ ]:
# Show a predetermined reference run (sinusoidal positions, seed 42).
# This avoids selecting a model based on test-set performance.
best_key = ("sinusoidal", 42)
yt, yp = taskA_predictions[best_key]
cm = confusion_matrix(yt, yp)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_title(f"Task A confusion matrix: {best_key}")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks(range(6)); ax.set_yticks(range(6))
for i in range(6):
    for j in range(6):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("problem2/figures/taskA_confusion_matrix.png", dpi=200)
plt.show()

**Comparison with Problem 1.** Use the best Problem 1 GRU result from the previous notebook as the recurrent reference. This notebook prints the Transformer results and parameter counts; enter the measured Problem 1 reference values in the comparison cell below so the comparison is grounded in your actual experiment rather than hard-coded assumptions.

In [ ]:
# Fill these from your verified Problem 1 Task 3 summary:
P1_BEST_MODEL = "GRU h=64"
P1_TEST_ACCURACY = 0.8932
P1_MACRO_F1 = 0.8937
P1_PARAMETERS = 14790
P1_SECONDS_PER_EPOCH = 0.1737  # measured average from the hidden-size sweep

comparison = taskA_summary.copy()
comparison["problem1_reference"] = (
    f"{P1_BEST_MODEL}: acc={P1_TEST_ACCURACY:.4f}, "
    f"macroF1={P1_MACRO_F1:.4f}, params={P1_PARAMETERS}"
)
display(comparison.round(4))

## 4. Task B — Tiny Shakespeare character-level causal language model

The dataset is split chronologically: first 90% for training and final 10% for validation, with no shuffling. The character vocabulary is built from the complete text. The model uses context length 128, width 128, 4 attention heads and 4 Transformer layers.

In [ ]:
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
shakespeare_path = Path("data/tinyshakespeare.txt")
if not shakespeare_path.exists():
    urllib.request.urlretrieve(SHAKESPEARE_URL, shakespeare_path)

text_corpus = shakespeare_path.read_text(encoding="utf-8")
vocab = sorted(set(text_corpus))
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}
encoded = torch.tensor([stoi[c] for c in text_corpus], dtype=torch.long)

split_index = int(0.9 * len(encoded))
train_tokens = encoded[:split_index]
val_tokens = encoded[split_index:]

print("Corpus characters:", len(text_corpus))
print("Vocabulary size:", len(vocab))
print("Train tokens:", len(train_tokens), "| Validation tokens:", len(val_tokens))
assert len(vocab) == 65, f"Expected 65 characters, found {len(vocab)}"

In [ ]:
class CharSequenceDataset(Dataset):
    def __init__(self, tokens, context):
        self.tokens = tokens
        self.context = context
    def __len__(self):
        return len(self.tokens) - self.context
    def __getitem__(self, idx):
        chunk = self.tokens[idx:idx+self.context+1]
        return chunk[:-1], chunk[1:]

def make_char_loader(tokens, context, batch_size, shuffle):
    ds = CharSequenceDataset(tokens, context)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      drop_last=True)

class CausalTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=128, num_heads=4,
                 num_layers=4, context=128, dropout=0.1,
                 use_causal_mask=True):
        super().__init__()
        self.context = context
        self.use_causal_mask = use_causal_mask
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Parameter(torch.zeros(1, context, d_model))
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, 4*d_model, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        nn.init.normal_(self.position_embedding, std=0.02)

    def forward(self, idx, need_weights=False):
        B, L = idx.shape
        assert L <= self.context
        x = self.token_embedding(idx) + self.position_embedding[:, :L]
        attn_weights = None
        # A lower-triangular mask prevents position t from seeing positions > t.
        causal_mask = None
        if self.use_causal_mask:
            causal_mask = torch.tril(
                torch.ones(L, L, dtype=torch.bool, device=idx.device)
            )
        for layer_idx, block in enumerate(self.blocks):
            if need_weights and layer_idx == 0:
                x, attn_weights = block(
                    x, need_weights=True, mask=causal_mask
                )
            else:
                x = block(x, mask=causal_mask)
        logits = self.lm_head(self.final_norm(x))
        if need_weights:
            return logits, attn_weights
        return logits

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=20):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context:]
            logits = self(idx_cond)[:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                k = min(top_k, logits.size(-1))
                values, _ = torch.topk(logits, k)
                logits[logits < values[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

### 4.1 Train and evaluate the causal language model

In [ ]:
def estimate_lm_loss(model, loader, max_batches=50):
    model.eval()
    losses = []
    with torch.no_grad():
        for i, (xb, yb) in enumerate(loader):
            if i >= max_batches: break
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            losses.append(F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), yb.reshape(-1)
            ).item())
    return float(np.mean(losses))

def train_lm(model, train_loader, val_loader, epochs, steps_per_epoch, seed):
    set_seed(seed)
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
    train_curve, val_curve = [], []
    for epoch in range(epochs):
        model.train()
        batch_losses = []
        train_iter = iter(train_loader)
        for step in range(steps_per_epoch):
            try:
                xb, yb = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                xb, yb = next(train_iter)
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                   yb.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            batch_losses.append(loss.item())
        train_loss = float(np.mean(batch_losses))
        val_loss = estimate_lm_loss(model, val_loader)
        train_curve.append(train_loss)
        val_curve.append(val_loss)
        print(f"Epoch {epoch+1}/{epochs}: train={train_loss:.4f}, val={val_loss:.4f}")
    return train_curve, val_curve

set_seed(SEEDS[0])
train_char_loader = make_char_loader(train_tokens, TASK_B_CONTEXT,
                                     TASK_B_BATCH_SIZE, shuffle=True)
val_char_loader = make_char_loader(val_tokens, TASK_B_CONTEXT,
                                   TASK_B_BATCH_SIZE, shuffle=False)

lm_model = CausalTransformerLM(
    vocab_size=len(vocab), d_model=D_MODEL, num_heads=TASK_B_HEADS,
    num_layers=TASK_B_LAYERS, context=TASK_B_CONTEXT,
    dropout=DROPOUT, use_causal_mask=True
).to(DEVICE)

print("Task B parameters:", count_parameters(lm_model))
lm_train_curve, lm_val_curve = train_lm(
    lm_model, train_char_loader, val_char_loader,
    epochs=TASK_B_EPOCHS,
    steps_per_epoch=TASK_B_STEPS_PER_EPOCH,
    seed=SEEDS[0]
)

final_val_nats = estimate_lm_loss(lm_model, val_char_loader, max_batches=100)
bits_per_character = final_val_nats / math.log(2)
print("Validation loss (nats/character):", final_val_nats)
print("Bits per character:", bits_per_character)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lm_train_curve, marker="o", label="Train")
ax.plot(lm_val_curve, marker="o", label="Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy (nats/character)")
ax.set_title("Tiny Shakespeare learning curve")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("problem2/figures/taskB_learning_curves.png", dpi=200)
plt.show()

### 4.2 Generate 500 characters at three temperatures

In [ ]:
prompt = "ROMEO:"
prompt_ids = torch.tensor([[stoi[c] for c in prompt]], dtype=torch.long, device=DEVICE)

generated_samples = {}
for temperature in [0.2, 0.8, 1.5]:
    generated = lm_model.generate(
        prompt_ids.clone(), max_new_tokens=500,
        temperature=temperature, top_k=20
    )[0].cpu().tolist()
    sample = "".join(itos[i] for i in generated)
    generated_samples[temperature] = sample
    print(f"\n{'='*25} temperature={temperature} {'='*25}\n")
    print(sample)

with open("problem2/generated_samples.txt", "w", encoding="utf-8") as f:
    for temp, sample in generated_samples.items():
        f.write(f"\n--- temperature={temp} ---\n{sample}\n")

## 5. Task 5 — Transformer analysis and ablations

### 5(a) Attention heatmaps for two heads

We visualise attention from two heads in the first Transformer block for one short input excerpt. Rows are query positions and columns are key positions. Since the model is causal, future positions should have zero attention.

In [ ]:
# Extract a short sequence from the validation corpus and inspect first-block attention.
heat_text = "ROMEO: But soft, what light through yonder window breaks?"
heat_ids = torch.tensor([[stoi[c] for c in heat_text]], dtype=torch.long, device=DEVICE)
lm_model.eval()
with torch.no_grad():
    x = lm_model.token_embedding(heat_ids) + lm_model.position_embedding[:, :heat_ids.size(1)]
    _, heat_weights = lm_model.blocks[0](x, need_weights=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, head in zip(axes, [0, 1]):
    mat = heat_weights[0, head].cpu().numpy()
    im = ax.imshow(mat, aspect="auto", origin="lower", cmap="viridis")
    ax.set_title(f"Attention head {head+1}")
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("Tiny Shakespeare — first-layer attention heads")
plt.tight_layout()
plt.savefig("problem2/figures/attention_two_heads.png", dpi=200)
plt.show()

### 5(b) Number-of-heads ablation: 1, 4 and 8

Keep the model width fixed at 128. To make the ablation practical in Colab, each head configuration is trained for a smaller, identical number of update steps. Compare validation loss and training time; this is an ablation comparison, not a replacement for the main 4-head model.

In [ ]:
HEAD_ABLATION_STEPS = 150
HEAD_ABLATION_EPOCHS = 2

def run_head_ablation(num_heads, seed=42):
    set_seed(seed)
    model = CausalTransformerLM(
        vocab_size=len(vocab), d_model=D_MODEL, num_heads=num_heads,
        num_layers=TASK_B_LAYERS, context=TASK_B_CONTEXT,
        dropout=DROPOUT, use_causal_mask=True
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    losses = []
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start = time.perf_counter()
    iterator = iter(train_char_loader)
    model.train()
    for step in range(HEAD_ABLATION_STEPS):
        try: xb, yb = next(iterator)
        except StopIteration:
            iterator = iter(train_char_loader); xb, yb = next(iterator)
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        peak_mb = torch.cuda.max_memory_allocated() / (1024**2)
    else:
        peak_mb = np.nan
    elapsed = time.perf_counter() - start
    val_loss = estimate_lm_loss(model, val_char_loader, max_batches=100)
    return {
        "heads": num_heads, "train_loss_last": float(np.mean(losses[-20:])),
        "val_loss_nats": val_loss, "seconds": elapsed,
        "peak_memory_mb": peak_mb, "params": count_parameters(model)
    }

head_ablation_results = [
    run_head_ablation(h, seed=SEEDS[0]) for h in [1, 4, 8]
]
head_ablation_df = pd.DataFrame(head_ablation_results)
display(head_ablation_df.round(4))
head_ablation_df.to_csv("problem2/head_ablation.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(
    head_ablation_df["heads"].to_numpy(),
    head_ablation_df["val_loss_nats"].to_numpy(),
    marker="o"
)

ax.set_xlabel("Number of attention heads")
ax.set_ylabel("Validation loss (nats/character)")
ax.set_title("Head-count ablation at fixed d=128")
ax.set_xticks([1, 4, 8])
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig("problem2/figures/head_ablation.png", dpi=200)
plt.show()

### 5(c) Remove the causal mask

A causal language model must not use future tokens when predicting the next character. This experiment trains an otherwise identical model without the causal mask. A lower teacher-forced loss can be misleading because each position can inspect future tokens. Generation quality is therefore also inspected.

In [ ]:
class UnmaskedTransformerBlock(TransformerEncoderBlock):
    # Same pre-LN block structure, but its MHA is naturally unmasked.
    pass

class AblationTransformerLM(CausalTransformerLM):
    def __init__(self, *args, use_causal_mask=True, **kwargs):
        super().__init__(*args, use_causal_mask=use_causal_mask, **kwargs)
        self.use_causal_mask = use_causal_mask

    def forward(self, idx, need_weights=False):
        B, L = idx.shape
        x = self.token_embedding(idx) + self.position_embedding[:, :L]
        attn_weights = None
        for layer_idx, block in enumerate(self.blocks):
            if not self.use_causal_mask:
                # Standard encoder self-attention has no triangular mask.
                if need_weights and layer_idx == 0:
                    x, attn_weights = block(x, need_weights=True)
                else:
                    x = block(x)
            else:
                # Apply explicit causal masking to every block.
                mask = torch.tril(torch.ones(L, L, dtype=torch.bool, device=idx.device))
                if need_weights and layer_idx == 0:
                    z = block.ln1(x)
                    a, attn_weights = block.attn(z, mask=mask, need_weights=True)
                    x = x + block.dropout(a)
                    x = x + block.dropout(block.ffn(block.ln2(x)))
                else:
                    z = block.ln1(x)
                    a = block.attn(z, mask=mask)
                    x = x + block.dropout(a)
                    x = x + block.dropout(block.ffn(block.ln2(x)))
        logits = self.lm_head(self.final_norm(x))
        if need_weights:
            return logits, attn_weights
        return logits

def train_mask_variant(use_mask, seed=42, steps=250):
    set_seed(seed)
    model = AblationTransformerLM(
        vocab_size=len(vocab), d_model=D_MODEL, num_heads=TASK_B_HEADS,
        num_layers=TASK_B_LAYERS, context=TASK_B_CONTEXT,
        dropout=DROPOUT, use_causal_mask=use_mask
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    iterator = iter(train_char_loader)
    model.train()
    losses = []
    start = time.perf_counter()
    for step in range(steps):
        try: xb, yb = next(iterator)
        except StopIteration:
            iterator = iter(train_char_loader); xb, yb = next(iterator)
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    val_loss = estimate_lm_loss(model, val_char_loader, max_batches=100)
    return model, float(np.mean(losses[-20:])), val_loss, time.perf_counter()-start

masked_model, masked_train, masked_val, masked_time = train_mask_variant(True)
unmasked_model, unmasked_train, unmasked_val, unmasked_time = train_mask_variant(False)

mask_comparison = pd.DataFrame([
    {"setting": "causal mask", "last_train_loss": masked_train,
     "validation_loss": masked_val, "seconds": masked_time},
    {"setting": "no causal mask", "last_train_loss": unmasked_train,
     "validation_loss": unmasked_val, "seconds": unmasked_time}
])
display(mask_comparison.round(4))
mask_comparison.to_csv("problem2/no_mask_ablation.csv", index=False)

# Generate from both models with the same prompt.
for label, model in [("causal", masked_model), ("no_mask", unmasked_model)]:
    ids = model.generate(prompt_ids.clone(), max_new_tokens=300,
                         temperature=0.8, top_k=20)[0].cpu().tolist()
    generated = "".join(itos[i] for i in ids)
    print(f"\n--- {label} model generation ---\n{generated}")
    Path(f"problem2/{label}_generation.txt").write_text(generated, encoding="utf-8")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(mask_comparison["setting"], mask_comparison["validation_loss"])
ax.set_ylabel("Validation loss (nats/character)")
ax.set_title("Effect of removing the causal mask")
plt.tight_layout()
plt.savefig("problem2/figures/no_mask_ablation.png", dpi=200)
plt.show()

### 5(d) Time and memory at context lengths 64, 128, 256 and 512

Self-attention forms an `n × n` score matrix per head, so attention compute and activation memory grow approximately as \(O(n^2 d)\) for fixed width and head count. We measure forward-plus-backward time and peak CUDA memory using the same model width, batch size and number of repetitions.

In [ ]:
def measure_context_cost(context, batch_size=8, repeats=3):
    set_seed(42)
    model = CausalTransformerLM(
        vocab_size=len(vocab), d_model=D_MODEL, num_heads=TASK_B_HEADS,
        num_layers=2, context=context, dropout=0.0,
        use_causal_mask=True
    ).to(DEVICE)
    x = torch.randint(0, len(vocab), (batch_size, context), device=DEVICE)
    y = torch.randint(0, len(vocab), (batch_size, context), device=DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    durations = []
    for _ in range(repeats):
        start = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        loss.backward()
        optimizer.step()
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        durations.append(time.perf_counter() - start)

    peak_mb = (torch.cuda.max_memory_allocated() / (1024**2)
               if DEVICE.type == "cuda" else np.nan)
    del model, optimizer, x, y, logits, loss
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return {"context": context, "seconds_per_step": float(np.mean(durations)),
            "peak_memory_mb": peak_mb}

context_results = pd.DataFrame([
    measure_context_cost(n) for n in [64, 128, 256, 512]
])
display(context_results.round(4))
context_results.to_csv("problem2/context_cost.csv", index=False)

fig, ax1 = plt.subplots(figsize=(7, 4))

# Convert Pandas Series to NumPy arrays before plotting
contexts = context_results["context"].to_numpy()
seconds = context_results["seconds_per_step"].to_numpy()
memory = context_results["peak_memory_mb"].to_numpy()

ax1.plot(contexts, seconds, marker="o",label="Seconds/step")

ax1.set_xlabel("Context length")
ax1.set_ylabel("Seconds per forward/backward step")
ax1.set_xticks([64, 128, 256, 512])

ax2 = ax1.twinx()
ax2.plot(contexts, memory, marker="s", linestyle="--",label="Peak CUDA memory")
ax2.set_ylabel("Peak CUDA memory (MB)")

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")

plt.title("Transformer cost versus context length")
fig.tight_layout()

plt.savefig("problem2/figures/context_time_memory.png", dpi=200,bbox_inches="tight")

plt.show()

## 6. Save results and final verification

The following cell writes Problem 2 experiment rows into the shared `results.csv` schema used in Problem 1. It also prints the generated artefacts. If you rerun the notebook, existing Problem 2 rows are replaced to avoid duplicate rows.

In [ ]:
RESULT_COLUMNS = [
    "problem", "model", "seed", "seq_len", "test_acc", "macro_f1",
    "test_mse", "params", "sec_per_epoch", "val_loss_nats",
    "bits_per_char", "notes"
]

p2_rows = list(taskA_results)
p2_rows.append({
    "problem": "P2", "model": "TinyShakespeare-Transformer",
    "seed": SEEDS[0], "seq_len": TASK_B_CONTEXT,
    "test_acc": np.nan, "macro_f1": np.nan, "test_mse": np.nan,
    "params": count_parameters(lm_model),
    "sec_per_epoch": np.nan, "val_loss_nats": final_val_nats,
    "bits_per_char": bits_per_character,
    "notes": f"Task B d={D_MODEL}, heads={TASK_B_HEADS}, layers={TASK_B_LAYERS}, epochs={TASK_B_EPOCHS}"
})
for _, row in head_ablation_df.iterrows():
    p2_rows.append({
        "problem": "P2", "model": f"TinyShakespeare-heads-{int(row['heads'])}",
        "seed": SEEDS[0], "seq_len": TASK_B_CONTEXT,
        "test_acc": np.nan, "macro_f1": np.nan, "test_mse": np.nan,
        "params": row["params"], "sec_per_epoch": np.nan,
        "val_loss_nats": row["val_loss_nats"], "bits_per_char": np.nan,
        "notes": "Task 5(b) head-count ablation"
    })
for _, row in mask_comparison.iterrows():
    p2_rows.append({
        "problem": "P2", "model": f"TinyShakespeare-{row['setting'].replace(' ', '-')}",
        "seed": SEEDS[0], "seq_len": TASK_B_CONTEXT,
        "test_acc": np.nan, "macro_f1": np.nan, "test_mse": np.nan,
        "params": count_parameters(masked_model if row["setting"] == "causal mask" else unmasked_model),
        "sec_per_epoch": np.nan, "val_loss_nats": row["validation_loss"],
        "bits_per_char": row["validation_loss"] / math.log(2),
        "notes": "Task 5(c) causal-mask ablation"
    })

p2_df = pd.DataFrame(p2_rows, columns=RESULT_COLUMNS)
results_path = Path("results.csv")
if results_path.exists():
    previous = pd.read_csv(results_path)
    if "problem" in previous.columns:
        previous = previous[previous["problem"] != "P2"]
    combined = pd.concat([previous, p2_df], ignore_index=True)
else:
    combined = p2_df
combined.to_csv(results_path, index=False)

print("Problem 2 result rows:", len(p2_df))
print("Combined results.csv rows:", len(combined))
print("\nSaved figures:")
for p in sorted(Path("problem2/figures").glob("*.png")):
    print(" -", p)
print("\nSaved tables/text:")
for p in sorted(Path("problem2").glob("*.csv")) + sorted(Path("problem2").glob("*.txt")):
    print(" -", p)
print("\nFinal key metrics:")
print("Task A summary:")
display(taskA_summary.round(4))
print("Task B validation loss:", round(final_val_nats, 4), "nats/character")
print("Task B bits/character:", round(bits_per_character, 4))